<!-- # Swin Transformer: Hierarchical Vision Transformer using Shifted Windows -->
# Swin Transformer: 使用变换窗口的分层视觉Transformer

- [论文](https://arxiv.org/abs/2103.14030)
- [代码](https://github.com/microsoft/Swin-Transformer)

## 1 引言

<div style="background-color: white; padding: 10px; border-radius: 5px; text-align: center; width: 60%; margin: auto;">
    <image src="./assets/teaser11.png" />
    <span style="color: black; font-size: 16px;"><strong>图 1</strong>：两种结构的特征图对比</span>
</div>

1. (a) 本文提出的`Swin Transformer`通过在更深层中合并图像分块（灰色部分所示）构建层级特征图；由于自注意力计算仅在每个局部窗口（红色部分所示）内进行，其计算复杂度与输入图像尺寸呈线性关系。因此，该模型可作为通用骨干网络，同时适用于图像分类和密集识别任务。
2. (b) 相比之下，以往的`ViT`模型仅生成单一低分辨率的特征图；由于自注意力采用全局计算方式，其计算复杂度与输入图像尺寸呈二次关系。

__Explanation:__  
1. 在ViT中，图像被划分为$N$个Patch。自注意力计算是($QK^T$)，即俩个$N\times d$矩阵相乘，计算复杂度为$O(N^2d)$。
2. 在SwinTransformer中，将尺寸$S$的图像分为$N$个Patch继续划分为多个窗口（Window），每个窗口包含$M\times M$个Patch。
   - 每个窗口内计算量：$O(M^4d)$
   - 窗口数量：$O(\frac{N}{M^2})$
   - 总计算量：$O(M^2Nd)$，即与输入图像尺寸线性

<div style="background-color: white; padding: 10px; border-radius: 5px; text-align: center; width: 60%; margin: auto;">
    <image src="./assets/teaser_v4.png" />
    <span style="color: black; font-size: 16px;"><strong>图 2</strong>：计算自注意力的移位窗口方法</span>
</div>

- 上述图片展示了如何通过移位窗口（Shifted Window）来实现跨窗口连接。
  - $l$层中，使用常规窗口划分方案，并在每个窗口内计算自注意。
  - 在$l+1$层中，窗口分区发生了偏移，新窗口自注意力计算跨越了前一层的边界，提供了跨窗口连接。

## 3 方法论

<div style="background-color: white; padding: 10px; border-radius: 5px; text-align: center; width: 60%; margin: auto;">
    <image src="./assets/HiT-arch-v2.png" />
    <span style="color: black; font-size: 16px;"><strong>图 3</strong>：Swin Transformer架构</span>
</div>

## 3.1 整体架构

以Sinw-T为例：图像($H\times W \times 3$)首先被划分为不重叠的$4\times 4$个Patch，每个Patch被视为一个$48$维度的token，接着通过线性层（`Linear Embedding`）投影到$C$维度。接着通过Swin Transformer Blocks进行特征提取，其中不改变特征图尺寸，这是第一阶段。第二阶段开始使用`Patch Merging`合并$2\times 2$邻域Patch（$\frac{H}{2}\times \frac{W}{2} \times 4C$），并通过线性层下采样到$2C$维度。（$\frac{H}{2}\times \frac{W}{2} \times 2C$），然后通过Swin Transformer Blocks进行特征提取，之后重复上述过程，形成第三阶段和第四阶段，输出特征图$\frac{H}{32}\times \frac{W}{32} \times 8C$。

__Swin Teansformer 块:__
1. 将Transformer中的标准多头自注意力层替换为基于位移窗口的多头自注意力层（`W-MSA`和`SW-MSA`），以实现线性计算复杂度。
2. `W-MSA`之后使用包含GELU的2层MLP，每个模块之前都适用LayerNorm，之后使用残差连接。



### 3.2 基于窗口移动的自注意力

标准的Transformer以及ViT变体都计算全局关注度，这导致$O(N^2)$的计算复杂度（$N$为token数量，在nlp中体现为序列长度，视觉中体现为图像尺寸）。不适合密集预测或高分辨率图像，文中使用移动窗口注意力来解决该问题。

__非重叠窗口中的自我关注:__
将图像分为多个不重叠的窗口，每个窗口包含$M\times M$个Patch，则图片$H\times W$（分为$h\times w$个图像Patch）在一个MSA内的计算复杂度（忽略SoftMax）：
$$\begin{align*}
\\
\Omega(\operatorname{MSA}) &= 4hw C^2 + 2(hw)^2C, \tag{1} \\
\Omega(\text{W-MSA}) &= 4hw C^2 + 2 M^2 hw C. \tag{2}
\end{align*}$$

__具体推导MSA:__  
1. 计算$Q,K,V$的复杂度：$[hw, C] \times [C, C] \times 3 = 3hwC^2$
2. 计算$QK^T$的复杂度：$[hw, C] \times [C, hw] = hw \times hw \times C = (hw)^2C$
3. 除以$\sqrt{d}$ + softmax：标量运算不计，Softmax逐行归一化，可忽略不计
4. 计算加权$V$：$[hw, hw] \times [hw, C] = hw \times hw \times C = (hw)^2C$
5. 最终线性变换：$[hw, C] \times [C, C] = hwC^2$
6. 总和：$\Omega(\operatorname{MSA}) = 4hwC^2 + 2(hw)^2C$


__具体推导W-MSA:__  
1. 计算$Q,K,V$的复杂度：$[hw, C] \times [C, C] \times 3 = 3hwC^2$
2. 窗口单独计算$QK^T$的复杂度：$\frac{hw}{M^2} \times [M^2, C] \times [C, M^2] = M^2hwC$
3. 除以$\sqrt{d}$ + softmax：标量运算不计，Softmax逐行归一化，可忽略不计
4. 窗口单独计算加权$V$：$\frac{hw}{M^2} \times [M^2, M^2] \times [M^2, C] = M^2hwC$
5. 最终线性变换：$[hw, C] \times [C, C] = hwC^2$
6. 总和：$\Omega(\operatorname{MSA}) = 4hwC^2 + 2M^2hwC$

__连续块的窗口分区位移：__

$$\begin{align*}
\hat{z}^l &= \text{W-MSA}(\operatorname{LN}(z^{l-1})) + z^{l-1}, \\
z^l &= \text{MLP}(\operatorname{LN}(\hat{z}^l)) + \hat{z}^l, \\
\hat{z}^{l+1} &= \text{SW-MSA}(\operatorname{LN}(z^l)) + z^l, \\
z^{l+1} &= \text{MLP}(\operatorname{LN}(\hat{z}^{l+1})) + \hat{z}^{l+1}. \tag{3}
\end{align*}$$

__针对移位配置的高效批处理计算：__

<div style="background-color: white; padding: 10px; border-radius: 5px; text-align: center; width: 60%; margin: auto;">
    <image src="./assets/HiT-layer2.png" />
    <span style="color: black; font-size: 16px;"><strong>图 4</strong>：移位窗口划分中高效批量计算方法的示例</span>
</div>

- 如[图 4]()所示，为了解决移位窗口划分后窗口数增加且窗口大小不一致的情况，文中提出了一种高效的批处理计算方法。即通过向左上角方向平移，将窗口拼接成大小一样的窗口，然后再用mask将注意力计算限制在原窗口内。

__相对位置偏置:__
- 作者在计算注意力相似度时添加了一个相对位置偏置项$B \in \mathbb{R}^{M^2 \times M^2}$
$$\begin{align*}
\operatorname{Attention}(Q, K, V) &= \operatorname{SoftMax}\left(\frac{QK^T}{\sqrt{d}} + B\right)V, \tag{4}
\end{align*}$$
- 其中$Q, K, V \in \mathbb{R}^{M^2 \times d}$分别为查询、键、值矩阵；$d$是查询/键的特征维度，$M^2$是窗口内Patch数量。
- 因为坐标轴向的相对位置范围为$[-M+1, M-1]$，所以我们只参数化一个尺寸更小的偏置矩阵$\hat{B} \in \mathbb{R}^{(2M-1) \times (2M-1)}$，从而构造$B$：
  - $B[i, j] = \hat{B}[x_i-x_j+1, y_i-y_j+1]$。
- 对比实验结果显示，使用相对位置偏置比绝对位置编码或者没有位置编码更有效。
- 预训练得到的相对位置偏执可以通过双三次插值初始化为不同窗口大小的模型微调。

### 3.3 模型架构变体

稳重构建了多种结构变体。基础模型`Swin-B`，大小和计算复杂度类似于`ViT-B/DeiT-B`。`Swin-T`和`Swin-S`的复杂度分别与`ResNet-50(DeiT-S)`和`ResNet-101`相似。默认$M=7$，每个头的查询维数为$32$，每个MLP的扩展层为$\alpha=4$。具体配置如下：

1. `Siwn-T`: $C=96$，layer numbers = {2, 2, 6, 2}
2. `Swin-S`: $C=96$，layer numbers = {2, 2, 18, 2}
3. `Swin-B`: $C=128$，layer numbers = {2, 2, 18, 2}
4. `Swin-L`: $C=192$，layer numbers = {2, 2, 18, 2}

## 4 实验

TODO: 待补充